In [ ]:
import os
import pickle
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
TRAIN_DATA_PATH = "f:/huyle_data_projects/stock_price_prediction/data/for_pretrained_models/meta_train/meta_train_merge.csv"
TEST_DATA_PATH  = "f:/huyle_data_projects/stock_price_prediction/data/for_pretrained_models/meta_test/meta_test_merge.csv"

FIGURES_DIR     = "f:/huyle_data_projects/stock_price_prediction/reports/figures/meta_ridge/"
MODEL_DIR       = "f:/huyle_data_projects/stock_price_prediction/models/meta_ridge/"
RESULTS_DIR     = "f:/huyle_data_projects/stock_price_prediction/data/for_pretrained_models/meta_ridge_predictions/"
SUMMARY_METRICS = "f:/huyle_data_projects/stock_price_prediction/reports/metrics/meta_ridge/Meta_Ridge_all_tickers.csv" # save metrics of meta-ridge on the 1 final test set

SYMBOLS = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
TARGET_COL = "Actual_Close"
FEATURE_COLS = [
    "ARIMAX_Predicted_Close", 
    "GRU_Predicted_Close", 
    "LSTM_Predicted_Close",
    "TRANSFORMER_Predicted_Close"
]
META_PRED_COL = "Meta_Predicted_Close"

In [ ]:
def calc_metrics(df: pd.DataFrame, actual_col: str = TARGET_COL, pred_col: str = META_PRED_COL, ticker_col: str = "Ticker") -> pd.DataFrame:
    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return df.groupby(ticker_col).apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None)).reset_index() # process each ticker separately and combine

    df = df.copy().sort_values("Date")
    df = df.dropna(subset=[actual_col, pred_col])
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) # add epsilon to avoid div by zero
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)

    mask  = actual_dir != 0
    tpa   = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan # calculate directional accuracy only for non-zero actual movements

    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE": mse, "MAE": mae, "MAPE": mape, "RMSE": rmse,
        "R2": r2, "DA": da, "TPA": tpa, "V-RMSE": v_rmse,
    }])

In [ ]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker  = df["Ticker"].iloc[0]
    df_plot = df.dropna(subset=[META_PRED_COL]).sort_values("Date")

    os.makedirs(save_dir, exist_ok=True)
    sns.set_theme(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    axes[0].plot(df_plot["Date"], df_plot["Close"], label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(df_plot["Date"], df_plot[META_PRED_COL], label="Meta Ridge Predicted", color="#ff7f0e", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"Meta Ridge Predicted Results: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis="x", rotation=30)

    error  = df_plot["Close"] - df_plot[META_PRED_COL]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")
    axes[1].bar(df_plot["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction Error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"meta_ridge_{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()

In [ ]:
def train_meta_ridge_one_ticker(ticker: str, df_train_all: pd.DataFrame, df_test_all: pd.DataFrame) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")
    
    df_train = df_train_all[df_train_all["Ticker"] == ticker].copy()
    df_test  = df_test_all[df_test_all["Ticker"] == ticker].copy()
    
    df_train = df_train.dropna(subset=FEATURE_COLS + [TARGET_COL]) # handle gaps caused by lookbacks in base models
    df_test  = df_test.dropna(subset=FEATURE_COLS + [TARGET_COL])
    
    log.info(f"  Train Meta: {len(df_train)} rows  Test Meta: {len(df_test)} rows")
    
    X_train = df_train[FEATURE_COLS]
    y_train = df_train[TARGET_COL]
    
    X_test = df_test[FEATURE_COLS]
    y_test = df_test[TARGET_COL]
    
    meta_model = Ridge(alpha=1.0, random_state=42)
    meta_model.fit(X_train, y_train)
    
    weights = ", ".join([f"{f}: {w:.3f}" for f, w in zip(FEATURE_COLS, meta_model.coef_)])
    log.info(f"  Model Weights | Intercept: {meta_model.intercept_:.3f} | {weights}")
    
    df_train[META_PRED_COL] = meta_model.predict(X_train)
    df_test[META_PRED_COL]  = meta_model.predict(X_test)
    
    train_metrics = calc_metrics(df_train)
    train_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Meta Train — MAE={train_metrics['MAE'].iloc[0]:.2f} DA={train_metrics['DA'].iloc[0]:.2%} R2={train_metrics['R2'].iloc[0]:.4f}")
    
    test_metrics = calc_metrics(df_test)
    test_metrics.insert(0, "Ticker", ticker)
    log.info(f"  Meta Test  — MAE={test_metrics['MAE'].iloc[0]:.2f} DA={test_metrics['DA'].iloc[0]:.2%} R2={test_metrics['R2'].iloc[0]:.4f}")
    
    # os.makedirs(METRICS_DIR, exist_ok=True)
    # test_metrics.to_csv(os.path.join(METRICS_DIR, f"Meta_Ridge_{ticker}_metrics.csv"), index=False)
    
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df_test.to_csv(os.path.join(RESULTS_DIR, f"Meta_Ridge_{ticker}_predictions.csv"), index=False)
    
    plot_results(df_test, save_dir=FIGURES_DIR)
    
    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"meta_ridge_{ticker}.pkl"), "wb") as f:
        pickle.dump({
            "model": meta_model,
            "ticker": ticker,
            "features": FEATURE_COLS
        }, f)
        
    log.info(f"  All outputs saved for {ticker}")
    return test_metrics

In [ ]:
if __name__ == "__main__":
    df_meta_train = pd.read_csv(TRAIN_DATA_PATH, parse_dates=["Date"])
    df_meta_test  = pd.read_csv(TEST_DATA_PATH, parse_dates=["Date"])
    
    all_metrics = []

    for ticker in SYMBOLS:
        try:
            m = train_meta_ridge_one_ticker(ticker, df_meta_train, df_meta_test)
            all_metrics.append(m)
        except Exception as e:
            log.error(f"{ticker} FAILED: {e}")

    if all_metrics:
        summary_df = pd.concat(all_metrics, ignore_index=True)
        os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
        summary_df.to_csv(SUMMARY_METRICS, index=False)
        log.info(f"Summary saved: {SUMMARY_METRICS}")
        print("\n" + summary_df.to_string(index=False))

    log.info("Done.")